<a href="https://colab.research.google.com/github/brigithsuarez-oss/SEMANA-2-/blob/main/TALLER_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from __future__ import annotations

import argparse
import shutil
import tempfile
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd


MONTH_NAMES_ES = {
    1: "Enero",
    2: "Febrero",
    3: "Marzo",
    4: "Abril",
    5: "Mayo",
    6: "Junio",
    7: "Julio",
    8: "Agosto",
    9: "Septiembre",
    10: "Octubre",
    11: "Noviembre",
    12: "Diciembre",
}
DATASET_URL = (
    "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
)
REQUIRED_COLUMNS = {
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country",
}

# Diccionario de normalización para mapear columnas alternativas del Excel original
COLUMN_MAPPING = {
    "customer id": "CustomerID",
    "customer_id": "CustomerID",
    "invoice": "InvoiceNo",
    "invoiceno": "InvoiceNo",
    "invoice no": "InvoiceNo",
    "price": "UnitPrice",
    "unitprice": "UnitPrice",
    "unit price": "UnitPrice",
    "stockcode": "StockCode",
    "stock code": "StockCode",
    "description": "Description",
    "quantity": "Quantity",
    "invoicedate": "InvoiceDate",
    "invoice date": "InvoiceDate",
    "country": "Country",
}


def download_dataset(data_dir: Path, force_download: bool = False) -> Path:
    """Descarga el ZIP de UCI y extrae el archivo Excel sin extraer otros archivos."""
    data_dir.mkdir(parents=True, exist_ok=True)
    archive_path = data_dir / "online_retail_ii.zip"
    workbook_path = data_dir / "online_retail_ii.xlsx"

    if workbook_path.exists() and not force_download:
        return workbook_path

    if archive_path.exists() and not zipfile.is_zipfile(archive_path):
        print("El ZIP existente está incompleto o dañado; se descargará nuevamente.")
        archive_path.unlink()

    if force_download or not archive_path.exists():
        last_error: Exception | None = None
        for attempt in range(1, 4):
            temporary_path: Path | None = None
            try:
                with tempfile.NamedTemporaryFile(
                    dir=data_dir,
                    prefix="online_retail_ii_",
                    suffix=".part",
                    delete=False,
                ) as temporary_file:
                    temporary_path = Path(temporary_file.name)
                    request = urllib.request.Request(
                        DATASET_URL,
                        headers={"User-Agent": "OnlineRetailII-data-loader/1.0"},
                    )
                    with urllib.request.urlopen(request, timeout=180) as response:
                        shutil.copyfileobj(response, temporary_file)

                if not zipfile.is_zipfile(temporary_path):
                    raise zipfile.BadZipFile("El archivo recibido no es un ZIP válido.")
                temporary_path.replace(archive_path)
                break
            except (urllib.error.URLError, TimeoutError, zipfile.BadZipFile) as error:
                last_error = error
                if attempt == 3:
                    raise RuntimeError(
                        "No se pudo descargar correctamente el conjunto de datos de UCI "
                        "tras 3 intentos. Comprueba tu conexión, proxy o firewall, o "
                        "descarga el ZIP desde la página de UCI y guárdalo como "
                        f"{archive_path}."
                    ) from error
                print(
                    f"Falló la descarga (intento {attempt}/3): {error}. "
                    f"Reintentando en {attempt * 5} segundos..."
                )
                time.sleep(attempt * 5)
            finally:
                if temporary_path is not None and temporary_path.exists():
                    temporary_path.unlink()

        if last_error is not None and not archive_path.exists():
            raise RuntimeError(
                f"No se pudo guardar un ZIP válido en {archive_path}."
            ) from last_error

    with zipfile.ZipFile(archive_path) as archive:
        workbook_members = [
            name for name in archive.namelist() if name.lower().endswith(".xlsx")
        ]
        if len(workbook_members) != 1:
            raise ValueError(
                "Se esperaba un solo archivo Excel dentro del ZIP de UCI; "
                f"se encontraron {len(workbook_members)}."
            )
        with archive.open(workbook_members[0]) as source:
            with workbook_path.open("wb") as destination:
                shutil.copyfileobj(source, destination)

    print(f"Archivo Excel listo: {workbook_path}")
    return workbook_path


def load_dataset(workbook_path: Path) -> pd.DataFrame:
    """Lee, normaliza columnas y combina las hojas anuales del libro Excel."""
    sheets = pd.read_excel(workbook_path, sheet_name=None, engine="openpyxl")
    if not sheets:
        raise ValueError(f"El archivo no contiene hojas de datos: {workbook_path}")

    frames = []
    for sheet_name, frame in sheets.items():
        # Normalizar los nombres de las columnas a minúsculas y sin espacios laterales
        frame.columns = [str(column).strip() for column in frame.columns]

        # Aplicar mapeo inteligente para corregir nombres de columnas desalineados (ej. 'Customer ID' -> 'CustomerID')
        normalized_columns = {}
        for col in frame.columns:
            col_lower = col.lower()
            if col_lower in COLUMN_MAPPING:
                normalized_columns[col] = COLUMN_MAPPING[col_lower]
            else:
                normalized_columns[col] = col
        frame = frame.rename(columns=normalized_columns)

        # Verificar que el resultado contenga todas las columnas obligatorias
        missing_columns = REQUIRED_COLUMNS.difference(frame.columns)
        if missing_columns:
            raise ValueError(
                f"Faltan columnas requeridas en la hoja {sheet_name!r}: "
                f"{', '.join(sorted(missing_columns))}. "
                f"Columnas actuales: {list(frame.columns)}"
            )
        frames.append(frame)

    return pd.concat(frames, ignore_index=True)


def clean_dataset(data: pd.DataFrame) -> pd.DataFrame:
    """Normaliza tipos, conserva devoluciones y calcula el importe por línea."""
    cleaned = data.copy()
    cleaned["InvoiceNo"] = (
        cleaned["InvoiceNo"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    )
    cleaned["StockCode"] = (
        cleaned["StockCode"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    )
    cleaned["Description"] = cleaned["Description"].astype("string").str.strip()
    cleaned["Country"] = cleaned["Country"].astype("string").str.strip()
    cleaned["Quantity"] = pd.to_numeric(cleaned["Quantity"], errors="coerce")
    cleaned["UnitPrice"] = pd.to_numeric(cleaned["UnitPrice"], errors="coerce")
    cleaned["InvoiceDate"] = pd.to_datetime(
        cleaned["InvoiceDate"], errors="coerce", dayfirst=True
    )

    customer_ids = pd.to_numeric(cleaned["CustomerID"], errors="coerce")
    cleaned["CustomerID"] = customer_ids.astype("Int64").astype("string")

    required_values = [
        "InvoiceNo",
        "StockCode",
        "Quantity",
        "UnitPrice",
        "InvoiceDate",
        "Country",
    ]
    cleaned = cleaned.dropna(subset=required_values).copy()
    cleaned["Description"] = cleaned["Description"].fillna("Unknown")
    cleaned["IsCancellation"] = cleaned["InvoiceNo"].str.upper().str.startswith("C")
    cleaned["TotalPrice"] = cleaned["Quantity"] * cleaned["UnitPrice" ]
    return cleaned.reset_index(drop=True)


def sales_by_country(data: pd.DataFrame) -> pd.DataFrame:
    """Resume cantidades e importes netos por país (incluye devoluciones)."""
    return (
        data.groupby("Country", as_index=False)
        .agg(
            Invoices=("InvoiceNo", "nunique"),
            Units=("Quantity", "sum"),
            Revenue=("TotalPrice", "sum"),
        )
        .sort_values("Revenue", ascending=False)
        .reset_index(drop=True)
    )


def monthly_sales(data: pd.DataFrame) -> pd.DataFrame:
    """Resume los importes netos por mes."""
    monthly = data.assign(Month=data["InvoiceDate"].dt.to_period("M").astype("string"))
    return (
        monthly.groupby("Month", as_index=False)
        .agg(Revenue=("TotalPrice", "sum"), Invoices=("InvoiceNo", "nunique"))
        .sort_values("Month")
        .reset_index(drop=True)
    )


def inspect_variables(data: pd.DataFrame) -> pd.DataFrame:
    """Resume el tipo, faltantes, valores únicos y ejemplos de cada variable."""
    rows = []
    for column in data.columns:
        values = data[column]
        non_missing = values.dropna()
        rows.append(
            {
                "Variable": column,
                "Tipo": str(values.dtype),
                "No_nulos": int(values.notna().sum()),
                "Faltantes": int(values.isna().sum()),
                "Porcentaje_faltantes": round(float(values.isna().mean() * 100), 2),
                "Valores_unicos": int(values.nunique(dropna=True)),
                "Ejemplos": " | ".join(non_missing.astype(str).unique()[:3]),
            }
        )
    return pd.DataFrame(rows)


def descriptive_analysis(data: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Genera estadísticas numéricas, categóricas y del periodo de fechas."""
    numeric_columns = data.select_dtypes(include="number").columns
    numeric_summary = data[numeric_columns].describe().transpose()
    if not numeric_summary.empty:
        numeric_summary["Mediana"] = data[numeric_columns].median()
        numeric_summary["Asimetria"] = data[numeric_columns].skew()
        numeric_summary.index.name = "Variable"
        numeric_summary = numeric_summary.reset_index()

    categorical_rows = []
    categorical_columns = data.select_dtypes(
        include=["object", "string", "category", "bool"]
    ).columns
    for column in categorical_columns:
        counts = data[column].value_counts(dropna=False).head(5)
        for value, frequency in counts.items():
            categorical_rows.append(
                {
                    "Variable": column,
                    "Valor": "<Faltante>" if pd.isna(value) else str(value),
                    "Frecuencia": int(frequency),
                    "Porcentaje": round(float(frequency / len(data) * 100), 2),
                }
            )
    categorical_summary = pd.DataFrame(
        categorical_rows,
        columns=["Variable", "Valor", "Frecuencia", "Porcentaje"],
    )

    date_columns = data.select_dtypes(include=["datetime", "datetimetz"]).columns
    date_rows = [
        {
            "Variable": column,
            "Fecha_minima": data[column].min(),
            "Fecha_maxima": data[column].max(),
            "Fechas_distintas": int(data[column].nunique()),
        }
        for column in date_columns
    ]
    date_summary = pd.DataFrame(
        date_rows,
        columns=["Variable", "Fecha_minima", "Fecha_maxima", "Fechas_distintas"],
    )
    return {
        "numeric": numeric_summary,
        "categorical": categorical_summary,
        "dates": date_summary,
    }


def save_descriptive_report(data: pd.DataFrame, data_dir: Path) -> None:
    """Guarda tablas CSV de inspección y análisis descriptivo."""
    data_dir.mkdir(parents=True, exist_ok=True)
    variables = inspect_variables(data)
    summaries = descriptive_analysis(data)

    variables.to_csv(data_dir / "resumen_variables.csv", index=False, encoding="utf-8-sig")
    summaries["numeric"].to_csv(
        data_dir / "estadisticas_numericas.csv", index=False, encoding="utf-8-sig"
    )
    summaries["categorical"].to_csv(
        data_dir / "resumen_categoricas.csv", index=False, encoding="utf-8-sig"
    )
    summaries["dates"].to_csv(
        data_dir / "resumen_fechas.csv", index=False, encoding="utf-8-sig"
    )

    print("\n=== Estructura y calidad de variables ===")
    print(variables.to_string(index=False))
    print("\n=== Estadísticas descriptivas numéricas ===")
    print(summaries["numeric"].to_string(index=False))
    print("\n=== Valores categóricos más frecuentes (máximo 5 por variable) ===")
    print(summaries["categorical"].to_string(index=False))
    if not summaries["dates"].empty:
        print("\n=== Cobertura de variables de fecha ===")
        print(summaries["dates"].to_string(index=False))
    print(f"\nInformes guardados en: {data_dir}")


def analyze_sales_patterns(
    data: pd.DataFrame, data_dir: Path, top_n: int = 10
) -> None:
    """Guarda distribuciones, productos líderes, tendencias y estacionalidad."""
    required_columns = {
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "UnitPrice",
        "InvoiceDate",
    }
    missing_columns = required_columns.difference(data.columns)
    if missing_columns:
        raise ValueError(
            "Faltan columnas necesarias para analizar las ventas: "
            f"{', '.join(sorted(missing_columns))}"
        )
    if top_n < 1:
        raise ValueError("top_n debe ser mayor que cero.")

    sales = data[
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "InvoiceDate",
        ]
    ].copy()
    sales["InvoiceDate"] = pd.to_datetime(sales["InvoiceDate"], errors="coerce")
    sales = sales.loc[
        ~sales["InvoiceNo"].astype("string").str.upper().str.startswith("C", na=False)
        & sales["Quantity"].gt(0)
        & sales["UnitPrice"].gt(0)
        & sales["InvoiceDate"].notna()
    ].copy()
    if sales.empty:
        raise ValueError(
            "No hay líneas de venta válidas después de excluir cancelaciones, "
            "devoluciones y valores faltantes."
        )

    sales["TotalPrice"] = sales["Quantity"] * sales["UnitPrice"]
    sales["Month"] = sales["InvoiceDate"].dt.to_period("M")
    sales["Year"] = sales["InvoiceDate"].dt.year
    sales["MonthNumber"] = sales["InvoiceDate"].dt.month

    product_summary = (
        sales.groupby("StockCode", as_index=False)
        .agg(
            UnitsSold=("Quantity", "sum"),
            Revenue=("TotalPrice", "sum"),
            Description=("Description", "first"),
        )
        .sort_values(["UnitsSold", "Revenue"], ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    product_summary.insert(0, "Rank", range(1, len(product_summary) + 1))
    top_codes = product_summary["StockCode"]

    monthly_products = (
        sales.loc[sales["StockCode"].isin(top_codes)]
        .groupby(["Month", "StockCode"], as_index=False)
        .agg(UnitsSold=("Quantity", "sum"), Revenue=("TotalPrice", "sum"))
        .sort_values("Month")
    )
    monthly_summary = (
        sales.groupby("Month", as_index=False)
        .agg(UnitsSold=("Quantity", "sum"), Revenue=("TotalPrice", "sum"))
        .sort_values("Month")
    )
    monthly_summary["Month"] = monthly_summary["Month"].astype("string")

    monthly_seasonality = (
        sales.groupby(["Month", "Year", "MonthNumber"], as_index=False)
        .agg(UnitsSold=("Quantity", "sum"), Revenue=("TotalPrice", "sum"))
        .sort_values("Month")
    )
    first_month = sales["Month"].min()
    last_month = sales["Month"].max()
    complete_months = monthly_seasonality.loc[
        monthly_seasonality["Month"].gt(first_month)
        & monthly_seasonality["Month"].lt(last_month)
    ]
    if complete_months.empty:
        raise ValueError(
            "Se necesitan al menos tres meses observados para analizar "
            "la estacionalidad excluyendo meses parciales."
        )
    seasonality = (
        complete_months.groupby("MonthNumber", as_index=False)
        .agg(
            AverageMonthlyRevenue=("Revenue", "mean"),
            MedianMonthlyRevenue=("Revenue", "median"),
            AverageUnitsSold=("UnitsSold", "mean"),
            YearsObserved=("Year", "nunique"),
        )
        .sort_values("MonthNumber")
    )
    seasonality.insert(
        1,
        "MonthName",
        seasonality["MonthNumber"].map(MONTH_NAMES_ES),
    )

    data_dir.mkdir(parents=True, exist_ok=True)
    product_summary.to_csv(
        data_dir / "productos_mas_vendidos.csv", index=False, encoding="utf-8-sig"
    )
    monthly_products.assign(Month=monthly_products["Month"].astype("string")).to_csv(
        data_dir / "tendencia_top_productos.csv",
        index=False,
        encoding="utf-8-sig",
    )
    monthly_summary.to_csv(
        data_dir / "ventas_mensuales.csv", index=False, encoding="utf-8-sig"
    )
    seasonality.to_csv(
        data_dir / "estacionalidad_mensual.csv", index=False, encoding="utf-8-sig"
    )

    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    figures_dir = data_dir / "graficos_ventas"
    figures_dir.mkdir(parents=True, exist_ok=True)

    quantity_limit = float(sales["Quantity"].quantile(0.99))
    price_limit = float(sales["UnitPrice"].quantile(0.99))
    visible_quantities = sales.loc[sales["Quantity"].le(quantity_limit), "Quantity"]
    visible_prices = sales.loc[sales["UnitPrice"].le(price_limit), "UnitPrice"]
    figure, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].hist(visible_quantities, bins=40, color="#4472C4", edgecolor="white")
    axes[0].set(
        title="Distribución de cantidad comprada",
        xlabel=f"Unidades (hasta percentil 99: {quantity_limit:.2f})",
        ylabel="Número de líneas de venta",
    )
    axes[1].hist(visible_prices, bins=40, color="#70AD47", edgecolor="white")
    axes[1].set(
        title="Distribución del precio unitario",
        xlabel=f"Precio unitario (£, hasta percentil 99: {price_limit:.2f})",
        ylabel="Número de líneas de venta",
    )
    figure.tight_layout()
    figure.savefig(figures_dir / "distribuciones_cantidad_precio.png", dpi=150)
    plt.close(figure)

    figure, axis = plt.subplots(figsize=(14, 7))
    for _, product in product_summary.iterrows():
        product_trend = monthly_products.loc[
            monthly_products["StockCode"].eq(product["StockCode"])
        ]
        label = f"{product['StockCode']} - {str(product['Description'])[:35]}"
        axis.plot(
            product_trend["Month"].astype("string"),
            product_trend["UnitsSold"],
            marker="o",
            markersize=3,
            linewidth=1.5,
            label=label,
        )
    axis.set(
        title=f"Tendencia mensual de unidades: {len(product_summary)} productos más vendidos",
        xlabel="Mes de compra",
        ylabel="Unidades vendidas",
    )
    axis.tick_params(axis="x", labelrotation=60)
    axis.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize="small")
    figure.tight_layout()
    figure.savefig(figures_dir / "tendencia_top_productos.png", dpi=150)
    plt.close(figure)

    figure, axis = plt.subplots(figsize=(14, 5))
    axis.plot(
        monthly_summary["Month"],
        monthly_summary["Revenue"],
        marker="o",
        color="#ED7D31",
    )
    axis.set(
        title="Ventas brutas mensuales",
        xlabel="Mes de compra",
        ylabel="Ventas (£)",
    )
    axis.tick_params(axis="x", labelrotation=60)
    figure.tight_layout()
    figure.savefig(figures_dir / "ventas_mensuales.png", dpi=150)
    plt.close(figure)

    figure, axis = plt.subplots(figsize=(12, 5))
    axis.bar(
        seasonality["MonthName"],
        seasonality["AverageMonthlyRevenue"],
        color="#5B9BD5",
    )
    axis.set(
        title="Estacionalidad: ventas mensuales promedio por mes calendario",
        xlabel="Mes",
        ylabel="Ventas promedio (£)",
    )
    axis.tick_params(axis="x", labelrotation=35)
    figure.tight_layout()
    figure.savefig(figures_dir / "estacionalidad_mensual.png", dpi=150)
    plt.close(figure)

    peak_period = monthly_seasonality.loc[
        monthly_seasonality["Revenue"].idxmax()
    ]
    peak_season = seasonality.loc[
        seasonality["AverageMonthlyRevenue"].idxmax()
    ]
    print("\n=== Productos más vendidos por unidades ===")
    print(product_summary.to_string(index=False))
    print("\n=== Estacionalidad por mes calendario (meses completos) ===")
    print(seasonality.to_string(index=False))
    print(
        "\nMes con mayor venta total observada: "
        f"{peak_period['Month']} (£{peak_period['Revenue']:,.2f})."
    )
    print(
        "Mes con mayor venta mensual promedio: "
        f"{peak_season['MonthName']} "
        f"(£{peak_season['AverageMonthlyRevenue']:,.2f})."
    )
    print(
        "\nPara facilitar la lectura, los histogramas muestran los valores "
        "hasta el percentil 99; los informes CSV conservan los datos agregados."
    )
    print(f"Tablas del análisis guardadas en: {data_dir}")
    print(f"Gráficos guardados en: {figures_dir}")


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Descarga Online Retail II de UCI y guarda una versión limpia."
    )
    parser.add_argument(
        "--data-dir",
        type=Path,
        default=Path("./data"),
        help="Carpeta donde guardar el ZIP, el Excel y el CSV.",
    )
    parser.add_argument(
        "--force-download",
        action="store_true",
        help="Vuelve a descargar el archivo aunque ya exista.",
    )
    parser.add_argument(
        "--analyze-clean",
        action="store_true",
        help="Analiza el CSV limpio existente sin descargar ni regenerar los datos.",
    )
    parser.add_argument(
        "--analyze-sales",
        action="store_true",
        help="Analiza productos, distribuciones y estacionalidad desde el CSV limpio.",
    )
    # Usamos una lista vacía para evitar que lea los argumentos internos del kernel en Jupyter
    import sys
    args = parser.parse_args([] if "ipykernel" in sys.modules else None)

    output_path = args.data_dir / "online_retail_ii_clean.csv"
    if args.analyze_clean or args.analyze_sales:
        if not output_path.exists():
            parser.error(
                f"No se encontró el CSV limpio: {output_path}. "
                "Ejecuta primero el script sin opciones de análisis."
            )
        data = pd.read_csv(output_path, parse_dates=["InvoiceDate"])
    else:
        workbook_path = download_dataset(args.data_dir, args.force_download)
        data = clean_dataset(load_dataset(workbook_path))
        data.to_csv(output_path, index=False, encoding="utf-8-sig")

    print(f"Filas limpias: {len(data):,}")
    print(f"CSV guardado en: {output_path}")
    if "IsCancellation" in data.columns:
        cancellations = data["IsCancellation"].astype("boolean")
        print(f"Cancelaciones: {int(cancellations.sum()):,}")
    print("\nVentas netas por país (primeros 10):")
    if {"Country", "InvoiceNo", "Quantity", "TotalPrice"}.issubset(data.columns):
        print(sales_by_country(data).head(10).to_string(index=False))
    print("\nVentas netas por mes (primeros 10):")
    if {"InvoiceDate", "InvoiceNo", "TotalPrice"}.issubset(data.columns):
        print(monthly_sales(data).head(10).to_string(index=False))
    save_descriptive_report(data, args.data_dir)
    if args.analyze_sales:
        analyze_sales_patterns(data, args.data_dir)


if __name__ == "__main__":
    # En entornos Jupyter interactivos no autoejecutamos main para evitar colisiones
    import sys
    if "ipykernel" not in sys.modules:
        main()


1. Mapeo y Normalización de Columnas (Consistencia Multianual)
El Problema: El conjunto de datos abarca dos periodos distintos (2009-2010 y 2010-2011). En el Excel original de UCI, las columnas tenían nombres inconsistentes entre hojas (por ejemplo, 'Customer ID' con espacio en una hoja y 'CustomerID' en otra, o 'Price' en lugar de 'UnitPrice').
La Solución: Creamos un diccionario inteligente de mapeo (COLUMN_MAPPING) que normalizó los nombres de las columnas a un estándar unificado (InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country) antes de concatenar la información.
2. Conversión Correcta de Tipos de Datos (Data Types)
InvoiceNo y StockCode: Se convirtieron a tipo de texto (string) explícito y se eliminaron espacios en blanco laterales. También se removió el sufijo .0 residual que a veces genera la lectura de números desde Excel.
InvoiceDate: Se parseó al formato estándar de fecha y hora (datetime64[ns]) interpretando correctamente el orden de los días.
CustomerID: Los identificadores de clientes venían con formato decimal (ej. 12345.0) o nulos. Se transformaron primero a números enteros (Int64) para remover el punto decimal y luego se guardaron como texto limpio.
Quantity y UnitPrice: Se aseguraron como numéricos puros utilizando pd.to_numeric con manejo de errores coercitivos.
3. Filtrado de Registros Críticos (Garantía de Calidad)
El algoritmo eliminó de manera estricta cualquier fila que tuviera valores nulos (NaN) en columnas esenciales para la facturación: InvoiceNo, StockCode, Quantity, UnitPrice, InvoiceDate y Country. Esto asegura que cada registro represente una transacción real evaluable.
4. Tratamiento de Descripciones Faltantes
Para no perder transacciones valiosas que únicamente carecían de texto explicativo en el producto, las descripciones nulas se completaron por defecto con el valor genérico 'Unknown' (Desconocido).
5. Creación de Variables de Valor Predictivo
IsCancellation (Booleana): Identifica si la transacción es una cancelación o devolución (aquellas cuyo InvoiceNo comienza con la letra 'C'). Esto es fundamental para análisis de comportamiento de clientes, cálculo de tasas de retorno y depuración de ingresos netos.
TotalPrice (Numérica): Se calculó de forma directa la multiplicación de Quantity * UnitPrice para cada línea de la factura. Esta columna es la base para predecir ventas (ventas brutas, valor de vida del cliente o LTV, y análisis de demanda monetaria).
La base resultante con 1,067,371 filas se guardó en data/online_retail_ii_clean.csv, lista y óptima para cualquier modelo predictivo (como pronósticos de series de tiempo de demanda o segmentación de clientes).

In [5]:
import sys
from pathlib import Path
import pandas as pd

# Definir el directorio de datos dentro del entorno de Colab
DATA_DIR = Path("./data")

print("1. Iniciando descarga del conjunto de datos (UCI Online Retail II)...")
# Descargar y extraer el archivo Excel
workbook_path = download_dataset(DATA_DIR, force_download=False)

print("\n2. Cargando datos desde el archivo Excel (este paso puede tomar un par de minutos)...")
# Cargar las hojas anuales usando las funciones corregidas
raw_data = load_dataset(workbook_path)

print(f"\n3. Limpiando y preparando el dataset. Filas originales: {len(raw_data):,}")
# Limpiar el dataset agregando columnas de análisis/predicción
clean_df = clean_dataset(raw_data)

# Guardar la base de datos limpia para análisis y futuros modelos de predicción
output_csv_path = DATA_DIR / "online_retail_ii_clean.csv"
clean_df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
print(f"-> Base de datos limpia guardada con éxito en: {output_csv_path}")
print(f"-> Filas resultantes listas para analizar/predecir: {len(clean_df):,}")

print("\n4. Generando reportes de calidad y estadísticas descriptivas...")
save_descriptive_report(clean_df, DATA_DIR)

print("\n=== Previsualización de los Datos Limpios ===")
display(clean_df.head())


1. Iniciando descarga del conjunto de datos (UCI Online Retail II)...

2. Cargando datos desde el archivo Excel (este paso puede tomar un par de minutos)...

3. Limpiando y preparando el dataset. Filas originales: 1,067,371
-> Base de datos limpia guardada con éxito en: data/online_retail_ii_clean.csv
-> Filas resultantes listas para analizar/predecir: 1,067,371

4. Generando reportes de calidad y estadísticas descriptivas...

=== Estructura y calidad de variables ===
      Variable           Tipo  No_nulos  Faltantes  Porcentaje_faltantes  Valores_unicos                                                                       Ejemplos
     InvoiceNo         string   1067371          0                  0.00           53628                                                       489434 | 489435 | 489436
     StockCode         string   1067371          0                  0.00            5304                                                        85048 | 79323P | 79323W
   Description         

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancellation,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,False,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,False,30.0


Este análisis cubrirá:

Distribución de variables clave: Visualización de cantidades de compra y precios unitarios filtrando valores extremos (percentil 99) para evitar distorsiones por valores atípicos (outliers).
Productos líderes y tendencias: Identificación de los 10 productos más vendidos y seguimiento de su demanda mes a mes.
Patrones de estacionalidad: Análisis de ingresos promedio por mes calendario y detección de los periodos de máximas ventas.
Procedo a generar la celda de análisis y visualización.

¡El análisis exploratorio y visual ha concluido con éxito! Aquí tienes la explicación detallada de los hallazgos clave basados en las variables, tendencias y estacionalidad de tus datos:

1. Distribución de Variables Clave (Cantidad y Precio)
Los histogramas generados han sido guardados en data/graficos_ventas/distribuciones_cantidad_precio.png.
Como se observa en las estadísticas del reporte, la gran mayoría de las transacciones individuales corresponden a cantidades pequeñas (con una mediana de 3 unidades por transacción) y a precios muy bajos (cercanos a la banda de £1.00 a £5.00).
El filtrado automático mediante percentiles (percentil 99) nos permite ignorar compras atípicas muy masivas (por ejemplo, transacciones industriales excepcionales de hasta 80,995 unidades) para visualizar el comportamiento natural del cliente minorista común sin distorsiones.
2. Productos Más Vendidos y sus Tendencias
El análisis identificó el Top 10 de productos líderes por cantidad acumulada vendida:

84077 (WORLD WAR 2 GLIDERS ASSTD DESIGNS): 110,138 unidades (£25,259.90)
85099B (JUMBO BAG RED WHITE SPOTTY): 98,349 unidades (£183,454.83) — ¡Gran generador de ingresos!
21212 (PACK OF 72 RETRO SPOT CAKE CASES): 96,560 unidades (£52,997.20)
85123A (WHITE HANGING HEART T-LIGHT HOLDER): 96,147 unidades (£263,109.67) — El producto estrella en ingresos netos de este grupo.
22197 (POPCORN HOLDER, SMALL): 89,898 unidades (£80,920.64)
La tendencia mensual detallada para cada uno de estos productos se ha trazado en la gráfica tendencia_top_productos.png en tu directorio, permitiendo ver picos específicos de demanda que puedes usar para alimentar modelos de predicción de inventario.

3. Patrones Estacionales y Períodos de Máximas Ventas
El análisis de meses completos revela una fuerte estacionalidad de fin de año:

El trimestre dorado (Q4): Las ventas promedio mensuales crecen exponencialmente a partir de septiembre:
Septiembre: ~£991,461.59 promedio.
Octubre: ~£1,160,231.61 promedio.
Noviembre (Pico Máximo): ~£1,489,884.41 promedio.
Diciembre: ~£1,262,598.79 promedio.
El mes récord: El mes con la mayor venta total jamás registrada en todo el dataset fue noviembre de 2011 con un total de £1,509,496.33 ingresados.
Temporada baja: Los meses de febrero (£538,485.81) y abril (£609,668.81) presentan los menores niveles de actividad comercial en promedio.
Archivos Listos para tus Predicciones:
Todo el set de datos limpio y estructurado junto con sus resúmenes están listos en tu entorno para que comiences a entrenar modelos predictivos (como previsión de demanda para el Q4, o clasificación de fugas de clientes):

Base de datos limpia: /content/data/online_retail_ii_clean.csv
Reportes tabulares de soporte: ventas_mensuales.csv, productos_mas_vendidos.csv y estacionalidad_mensual.csv en la carpeta /content/data/.

In [6]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

# Ejecutar el análisis de patrones de ventas (esto genera los CSVs y los gráficos en el disco)
print("Ejecutando análisis de patrones de ventas...")
analyze_sales_patterns(clean_df, DATA_DIR, top_n=10)

# Ruta donde se guardaron las imágenes
graficos_dir = DATA_DIR / "graficos_ventas"

# Mostrar las imágenes generadas directamente en el Notebook
print("\n=== VISUALIZACIÓN DE RESULTADOS Y GRÁFICOS ===")

# 1. Distribución de cantidad y precio
if (graficos_dir / "distribuciones_cantidad_precio.png").exists():
    plt.figure(figsize=(12, 6))
    img = mpimg.imread(graficos_dir / "distribuciones_cantidad_precio.png")
    plt.imshow(img)
    plt.axis('off')
    plt.title("1. Distribuciones de Cantidad y Precio Unitario", fontsize=14, pad=10)
    plt.show()

# 2. Tendencia mensual de los top productos
if (graficos_dir / "tendencia_top_productos.png").exists():
    plt.figure(figsize=(12, 6))
    img = mpimg.imread(graficos_dir / "tendencia_top_productos.png")
    plt.imshow(img)
    plt.axis('off')
    plt.title("2. Tendencia Mensual de Unidades para los Top 10 Productos", fontsize=14, pad=10)
    plt.show()

# 3. Ventas mensuales brutas
if (graficos_dir / "ventas_mensuales.png").exists():
    plt.figure(figsize=(12, 6))
    img = mpimg.imread(graficos_dir / "ventas_mensuales.png")
    plt.imshow(img)
    plt.axis('off')
    plt.title("3. Evolución del Ingreso Mensual Bruto", fontsize=14, pad=10)
    plt.show()

# 4. Estacionalidad mensual promedio
if (graficos_dir / "estacionalidad_mensual.png").exists():
    plt.figure(figsize=(12, 6))
    img = mpimg.imread(graficos_dir / "estacionalidad_mensual.png")
    plt.imshow(img)
    plt.axis('off')
    plt.title("4. Estacionalidad: Promedio de Ventas por Mes Calendario", fontsize=14, pad=10)
    plt.show()


Ejecutando análisis de patrones de ventas...

=== Productos más vendidos por unidades ===
 Rank StockCode  UnitsSold   Revenue                        Description
    1     84077     110138  25259.90  WORLD WAR 2 GLIDERS ASSTD DESIGNS
    2    85099B      98349 183454.83         JUMBO BAG RED WHITE SPOTTY
    3     21212      96560  52997.20   PACK OF 72 RETRO SPOT CAKE CASES
    4    85123A      96147 263109.67 WHITE HANGING HEART T-LIGHT HOLDER
    5     22197      89898  80920.64             POPCORN HOLDER , SMALL
    6     84879      81809 132187.92      ASSORTED COLOUR BIRD ORNAMENT
    7     23843      80995 168469.60        PAPER CRAFT , LITTLE BIRDIE
    8     23166      78033  81700.92     MEDIUM CERAMIC TOP STORAGE JAR
    9     17003      71430  14959.19                 BROCADE RING PURSE
   10     21977      56794  28490.98 PACK OF 60 PINK PAISLEY CAKE CASES

=== Estacionalidad por mes calendario (meses completos) ===
 MonthNumber  MonthName  AverageMonthlyRevenue  MedianMon

**Parte 1: Clasificación de clientes:**
Definir una nueva columna en los datos para trabajar con dos clases: “Clientes
Normales” y “Clientes Premium”. ¿Cómo podríamos definir estas categorías, por
volumen de compras, cantidades, etc? Definir este campo.

Entrenar un modelo de clasificación para la clasificación de los clientes en estas
dos categorías.

In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

# 1. AGREGACIÓN DE DATOS A NIVEL DE CLIENTE (Métricas RFM)
# Excluimos transacciones que no tengan CustomerID
df_customers = clean_df.dropna(subset=['CustomerID']).copy()

# Definimos la fecha de referencia para la recencia (un día después del último registro)
max_date = df_customers['InvoiceDate'].max() + pd.Timedelta(days=1)

# Agregamos por cliente
print("Calculando métricas agregadas por cliente...")
customer_metrics = df_customers.groupby('CustomerID').agg(
    Monetary=('TotalPrice', 'sum'),
    Frequency=('InvoiceNo', 'nunique'),
    QuantityTotal=('Quantity', 'sum'),
    AverageUnitPrice=('UnitPrice', 'mean'),
    Recency=('InvoiceDate', lambda x: (max_date - x.max()).days)
).reset_index()

# 2. DEFINICIÓN DE CLASES: CLIENTE PREMIUM (Top 20% en Monetary o Frequency)
monetary_threshold = customer_metrics['Monetary'].quantile(0.80)
frequency_threshold = customer_metrics['Frequency'].quantile(0.80)

print(f"Umbral de ingresos para ser Premium (Top 20%): £{monetary_threshold:,.2f}")
print(f"Umbral de número de transacciones para ser Premium (Top 20%): {frequency_threshold:.0f} facturas")

customer_metrics['IsPremium'] = np.where(
    (customer_metrics['Monetary'] >= monetary_threshold) | (customer_metrics['Frequency'] >= frequency_threshold),
    1, # Premium
    0  # Normal
)

# Verificar la distribución de clases
class_counts = customer_metrics['IsPremium'].value_counts()
print(f"\nDistribución de Clases:\nNormales (0): {class_counts[0]} ({class_counts[0]/len(customer_metrics)*100:.1f}%)")
print(f"Premium (1): {class_counts[1]} ({class_counts[1]/len(customer_metrics)*100:.1f}%)")


Calculando métricas agregadas por cliente...
Umbral de ingresos para ser Premium (Top 20%): £2,858.66
Umbral de número de transacciones para ser Premium (Top 20%): 10 facturas

Distribución de Clases:
Normales (0): 4438 (74.7%)
Premium (1): 1504 (25.3%)


In [8]:
# 3. ENTRENAMIENTO DEL MODELO DE CLASIFICACIÓN
# Usaremos variables derivadas del comportamiento de compra inicial para predecir si es premium o normal
# Features: Cantidad de artículos comprados, precio unitario promedio de sus elecciones y recencia
# (Excluimos Frequency y Monetary directos ya que se usaron para crear la etiqueta objetivo y causarían data leakage)
X = customer_metrics[['QuantityTotal', 'AverageUnitPrice', 'Recency']]
y = customer_metrics['IsPremium']

# División de datos (Train / Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("Entrenando clasificador Random Forest...")
clf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=8)
clf.fit(X_train, y_train)

# Predicciones
y_pred = clf.predict(X_test)

# 4. EVALUACIÓN DEL MODELO
accuracy = accuracy_score(y_test, y_pred)
print(f"\nPrecisión Global (Accuracy): {accuracy * 100:.2f}%")
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=['Cliente Normal', 'Cliente Premium']))

# Graficar Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Premium'],
            yticklabels=['Normal', 'Premium'])
plt.title('Matriz de Confusión - Clasificación de Clientes')
plt.ylabel('Clase Real')
plt.xlabel('Clase Predicha')
plt.tight_layout()
plt.show()


Entrenando clasificador Random Forest...

Precisión Global (Accuracy): 92.65%

Reporte de Clasificación:
                 precision    recall  f1-score   support

 Cliente Normal       0.94      0.96      0.95      1332
Cliente Premium       0.88      0.82      0.85       451

       accuracy                           0.93      1783
      macro avg       0.91      0.89      0.90      1783
   weighted avg       0.93      0.93      0.93      1783



**PARTE 2**
 Segmentación de Clientes


Usar el agrupamiento K-means para categorizar a los clientes en grupos distintos
y analizar las características de cada segmento.

Usar el agrupamiento Mean Shift para categorizar a los clientes en grupos
distintos y analizar las características de cada segmento.

Comparar los resultados de las dos técnicas anteriores



In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, MeanShift, estimate_bandwidth

# 1. PREPARACIÓN Y ESTANDARIZACIÓN DE LOS DATOS
# Usamos las métricas clave agregadas por cliente
features = ['QuantityTotal', 'AverageUnitPrice', 'Recency']
X_clustering = customer_metrics[features].copy()

# Aplicamos una transformación logarítmica para mitigar el sesgo extremo de los valores atípicos antes de escalar
# Añadimos una constante pequeña para manejar valores negativos o nulos que pudieran existir en devoluciones netas
X_log = np.log1p(X_clustering - X_clustering.min())

# Estandarización para que todas las variables tengan media 0 y varianza 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

# 2. AGRUPAMIENTO K-MEANS
print("Ejecutando K-means con 4 segmentos...")
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
customer_metrics['Cluster_KMeans'] = kmeans.fit_predict(X_scaled)

# 3. AGRUPAMIENTO MEAN SHIFT
print("Estimando el ancho de banda y ejecutando Mean Shift...")
# Estimamos un ancho de banda razonable para el algoritmo
bandwidth = estimate_bandwidth(X_scaled, quantile=0.2, n_samples=1000, random_state=42)
# Si el ancho de banda estimado es nulo o demasiado pequeño, usamos uno por defecto
if bandwidth <= 0:
    bandwidth = 1.0

meanshift = MeanShift(bandwidth=bandwidth, bin_seeding=True)
customer_metrics['Cluster_MeanShift'] = meanshift.fit_predict(X_scaled)
num_ms_clusters = customer_metrics['Cluster_MeanShift'].nunique()
print(f"-> Mean Shift identificó automáticamente {num_ms_clusters} clústeres basados en la densidad.")

# 4. ANÁLISIS Y CARACTERÍSTICAS DE CADA SEGMENTO EN K-MEANS
print("\n=== Características Promedio por Segmento (K-Means) ===")
kmeans_summary = customer_metrics.groupby('Cluster_KMeans')[features + ['Monetary', 'Frequency']].mean()
display(kmeans_summary.round(2))

# 5. ANÁLISIS Y CARACTERÍSTICAS DE CADA SEGMENTO EN MEAN SHIFT
print("\n=== Características Promedio por Segmento (Mean Shift - Top 5 grupos por tamaño) ===")
ms_sizes = customer_metrics['Cluster_MeanShift'].value_counts()
ms_summary = customer_metrics.groupby('Cluster_MeanShift')[features + ['Monetary', 'Frequency']].mean()
ms_summary['Size'] = ms_sizes
display(ms_summary.sort_values(by='Size', ascending=False).head(5).round(2))


Ejecutando K-means con 4 segmentos...
Estimando el ancho de banda y ejecutando Mean Shift...
-> Mean Shift identificó automáticamente 17 clústeres basados en la densidad.

=== Características Promedio por Segmento (K-Means) ===


,QuantityTotal,AverageUnitPrice,Recency,Monetary,Frequency
Cluster_KMeans,,,,,
0,361.90,1233.05,464.80,-588.22,2.71
1,881.46,3.24,35.49,1565.10,6.34
2,405.74,3.66,367.67,734.91,3.07
3,9409.18,3.44,36.22,14754.80,28.86



=== Características Promedio por Segmento (Mean Shift - Top 5 grupos por tamaño) ===


,QuantityTotal,AverageUnitPrice,Recency,Monetary,Frequency,Size
Cluster_MeanShift,,,,,,
0,898.91,3.35,207.08,1570.12,5.72,5580
2,14312.33,2.85,10.47,22858.95,46.57,221
1,91.09,67.59,403.07,297.04,2.28,58
3,25.57,663.57,529.14,-791.24,1.95,21
8,3708.86,29.30,47.57,5457.08,16.50,14


In [10]:
# 6. VISUALIZACIÓN COMPARATIVA DE LAS DOS TÉCNICAS
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Gráfico 1: K-Means
sns.scatterplot(
    data=customer_metrics,
    x='QuantityTotal',
    y='Recency',
    hue='Cluster_KMeans',
    palette='tab10',
    ax=axes[0],
    alpha=0.6
)
axes[0].set_title('Segmentación de Clientes: K-Means (K=4)')
axes[0].set_xscale('log')
axes[0].set_xlabel('Cantidad Total de Artículos (Escala Log)')
axes[0].set_ylabel('Recencia (Días sin comprar)')

# Gráfico 2: Mean Shift
sns.scatterplot(
    data=customer_metrics,
    x='QuantityTotal',
    y='Recency',
    hue='Cluster_MeanShift',
    palette='viridis',
    ax=axes[1],
    alpha=0.6
)
axes[1].set_title(f'Segmentación de Clientes: Mean Shift (K={num_ms_clusters})')
axes[1].set_xscale('log')
axes[1].set_xlabel('Cantidad Total de Artículos (Escala Log)')
axes[1].set_ylabel('Recencia (Días sin comprar)')

plt.tight_layout()
plt.show()

# 7. COMPARACIÓN NUMÉRICA Y CONCLUSIONES
print("\n=== Comparación de Distribución de Clientes ===")
print("Tamaño de Clústeres en K-Means:")
print(customer_metrics['Cluster_KMeans'].value_counts().sort_index())

print("\nTamaño de Clústeres en Mean Shift:")
print(customer_metrics['Cluster_MeanShift'].value_counts())



=== Comparación de Distribución de Clientes ===
Tamaño de Clústeres en K-Means:
Cluster_KMeans
0      79
1    2220
2    2891
3     752
Name: count, dtype: int64

Tamaño de Clústeres en Mean Shift:
Cluster_MeanShift
0     5580
2      221
1       58
3       21
8       14
6       11
4       11
14      10
5        4
10       2
11       2
7        2
9        2
12       1
16       1
15       1
13       1
Name: count, dtype: int64


**PARTE 3**

Predicción de Ventas

● Crear nuevas características, si es necesario, para ayudar en el modelado
predictivo (por ejemplo, época del año, categorías de productos).

● Dividir los datos en conjuntos de entrenamiento y prueba.

● Elegir un algoritmo de aprendizaje automático adecuado (por ejemplo, regresión
lineal, árbol de decisión o bosque aleatorio) para predecir ventas futuras.

● Evaluar el modelo utilizando métricas de rendimiento apropiadas


In [11]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

# 1. AGREGACIÓN DE VENTAS DIARIAS
print("Agregando las transacciones a nivel diario para el modelo predictivo...")
# Excluimos cancelaciones para modelar el flujo de ventas regular y limpio
sales_only = clean_df[~clean_df['IsCancellation']].copy()
sales_only['Date'] = sales_only['InvoiceDate'].dt.date

daily_sales = sales_only.groupby('Date').agg(
    TotalRevenue=('TotalPrice', 'sum'),
    TotalQuantity=('Quantity', 'sum'),
    InvoiceCount=('InvoiceNo', 'nunique')
).reset_index()

daily_sales['Date'] = pd.to_datetime(daily_sales['Date'])
daily_sales = daily_sales.sort_values('Date').reset_index(drop=True)

# 2. INGENIERÍA DE CARACTERÍSTICAS (FEATURES TEMPORALES)
daily_sales['Month'] = daily_sales['Date'].dt.month
daily_sales['DayOfWeek'] = daily_sales['Date'].dt.dayofweek
daily_sales['DayOfYear'] = daily_sales['Date'].dt.dayofyear
daily_sales['IsWeekend'] = daily_sales['DayOfWeek'].isin([5, 6]).astype(int)

# Definir la época del año o estacionalidad (ej. Q4 es de alta demanda)
def get_season(month):
    if month in [11, 12]:
        return 3 # 'Peak / Holidays'
    elif month in [9, 10]:
        return 2 # 'High / Autumn'
    elif month in [3, 5, 6, 8]:
        return 1 # 'Medium / Spring-Summer'
    else:
        return 0 # 'Low'

daily_sales['SeasonCategory'] = daily_sales['Month'].apply(get_season)

# Crear variables de rezago (Lag Features) para capturar la tendencia de días anteriores
daily_sales['Revenue_Lag1'] = daily_sales['TotalRevenue'].shift(1)
daily_sales['Revenue_Lag7'] = daily_sales['TotalRevenue'].shift(7)
daily_sales['Revenue_RollMean7'] = daily_sales['TotalRevenue'].shift(1).rolling(window=7).mean()

# Eliminamos las filas con valores nulos generados por las variables de rezago
daily_sales = daily_sales.dropna().reset_index(drop=True)

display(daily_sales.head())


Agregando las transacciones a nivel diario para el modelo predictivo...


,Date,TotalRevenue,TotalQuantity,InvoiceCount,Month,DayOfWeek,DayOfYear,IsWeekend,SeasonCategory,Revenue_Lag1,Revenue_Lag7,Revenue_RollMean7
0,2009-12-08,49517.23,25695,114,12,1,342,0,3,45083.35,54513.50,44590.982857
1,2009-12-09,40616.09,17632,104,12,2,343,0,3,49517.23,63352.51,43877.230000
2,2009-12-10,44442.11,23523,95,12,3,344,0,3,40616.09,74037.91,40629.170000
3,2009-12-11,39659.60,19817,73,12,4,345,0,3,44442.11,40732.92,36401.198571
4,2009-12-13,22313.09,12611,71,12,6,347,1,3,39659.60,9803.05,36247.867143


In [12]:
# 3. DIVISIÓN CRONOLÓGICA (ENTRENAMIENTO Y PRUEBA)
# Para series temporales, entrenamos con el pasado y evaluamos en el futuro reciente
# Usaremos el último 20% de los días como conjunto de prueba
split_idx = int(len(daily_sales) * 0.8)

train_data = daily_sales.iloc[:split_idx]
test_data = daily_sales.iloc[split_idx:]

features_reg = ['Month', 'DayOfWeek', 'DayOfYear', 'IsWeekend', 'SeasonCategory', 'Revenue_Lag1', 'Revenue_Lag7', 'Revenue_RollMean7']
target_reg = 'TotalRevenue'

X_train_reg = train_data[features_reg]
y_train_reg = train_data[target_reg]
X_test_reg = test_data[features_reg]
y_test_reg = test_data[target_reg]

print(f"Registros de entrenamiento: {len(X_train_reg)} días ({train_data['Date'].min().date()} a {train_data['Date'].max().date()})")
print(f"Registros de prueba: {len(X_test_reg)} días ({test_data['Date'].min().date()} a {test_data['Date'].max().date()})")

# 4. SELECCIÓN Y ENTRENAMIENTO DEL ALGORITMO (Random Forest Regressor)
print("Entrenando el modelo RandomForestRegressor...")
regressor = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
regressor.fit(X_train_reg, y_train_reg)

# Predicciones
y_pred_reg = regressor.predict(X_test_reg)

# 5. EVALUACIÓN DEL MODELO
mae = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2 = r2_score(y_test_reg, y_pred_reg)
mean_val = y_test_reg.mean()

print("\n=== Métricas de Rendimiento del Modelo ===")
print(f"Error Absoluto Medio (MAE): £{mae:,.2f}")
print(f"Raíz del Error Cuadrático Medio (RMSE): £{rmse:,.2f}")
print(f"Coeficiente de Determinación (R²): {r2:.4f}")
print(f"Venta diaria promedio en Test: £{mean_val:,.2f} (Error MAE de {mae/mean_val*100:.2f}%)")

# 6. VISUALIZACIÓN DE LA PREDICCIÓN
plt.figure(figsize=(15, 6))
plt.plot(train_data['Date'], train_data['TotalRevenue'], label='Histórico de Entrenamiento', color='#4472C4', alpha=0.5)
plt.plot(test_data['Date'], test_data['TotalRevenue'], label='Venta Real de Prueba', color='#70AD47', linewidth=2)
plt.plot(test_data['Date'], y_pred_reg, label='Predicción del Modelo', color='#ED7D31', linestyle='--', linewidth=2)
plt.title('Predicción de Ventas Diarias con Random Forest')
plt.xlabel('Fecha')
plt.ylabel('Ventas Totales (£)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


Registros de entrenamiento: 477 días (2009-12-08 a 2011-07-21)
Registros de prueba: 120 días (2011-07-22 a 2011-12-09)
Entrenando el modelo RandomForestRegressor...

=== Métricas de Rendimiento del Modelo ===
Error Absoluto Medio (MAE): £13,026.76
Raíz del Error Cuadrático Medio (RMSE): £20,114.13
Coeficiente de Determinación (R²): 0.3727
Venta diaria promedio en Test: £44,406.67 (Error MAE de 29.34%)
